# Session 4 · Part One — Data That Survives

**Information Systems Development (SE)** · Dr. Pejman Ebrahimi · University of Liechtenstein

---

Everything your programs have remembered so far lived in **memory**. Close the program
and it is gone.

Today it stops being gone.

### How this notebook works

Run the cells **in order**, top to bottom, with **Shift + Enter**.

Every file used here is **created by the notebook itself**, in a folder called `demo/`.
Nothing is missing, nothing needs downloading. Open `demo/` in the VS Code explorer on the
left and watch the files appear as we go.

Some cells are **meant to crash**. They are labelled like this:

```
# THIS CELL IS MEANT TO FAIL
```

When one of those raises a red traceback, that is the lesson, not a mistake.

---
## 0 · A clean workspace

Run this first. It makes an empty `demo/` folder beside the notebook.

Run it again at any time to wipe everything and start over.

In [ ]:
import shutil, os
from pathlib import Path

DEMO = Path("demo")

if DEMO.exists():
    shutil.rmtree(DEMO)
DEMO.mkdir()

print(f"Clean workspace ready: {DEMO.resolve()}")
print("Files inside:", list(DEMO.iterdir()) or "(none yet)")

In [ ]:
def show(path):
    """Print a file exactly as it sits on disk. We will use this a lot."""
    p = Path(path)
    if not p.exists():
        print(f"--- {p} does not exist ---")
        return
    print(f"--- {p}  ({p.stat().st_size} bytes) ---")
    print(p.read_text(encoding="utf-8"))
    print("--- end ---")


def ls():
    """List what is in the demo folder right now."""
    items = sorted(DEMO.iterdir())
    if not items:
        print("demo/ is empty")
    for p in items:
        print(f"  {p.name:22} {p.stat().st_size:>6} bytes")


ls()

---
# 1 · Memory forgets. Disk does not.

A variable lives in memory. The moment the program ends, it is gone.

In [ ]:
basket = []
basket.append("Laptop")
basket.append("Mouse")

print(basket)
print("This list exists right now, and only right now.")

Restart the kernel (**Kernel → Restart**) and run the cell above again — `basket` is empty,
because the whole program started over.

Now the same two items, written to a file.

In [ ]:
with open(DEMO / "basket.txt", "w") as f:
    f.write("Laptop\n")
    f.write("Mouse\n")

show(DEMO / "basket.txt")

That file is on your disk. Restart the kernel, shut the laptop, come back on Monday —
it is still there.

**That is the whole idea of today.**

---
# 2 · Text files

## 2.1 Three ways to open a file

| Mode | What it does |
|---|---|
| `"r"` | **read** — the default. Fails if the file is missing. |
| `"w"` | **write** — creates the file, and **erases** it if it already exists. |
| `"a"` | **append** — adds to the end, keeps what was there. |

Let us prove that `"w"` really does erase.

In [ ]:
path = DEMO / "modes.txt"

with open(path, "w") as f:
    f.write("first line\n")
    f.write("second line\n")

show(path)

In [ ]:
# open it again with "w" and write only ONE short line
with open(path, "w") as f:
    f.write("oops\n")

show(path)      # the two lines above are gone forever

Both lines disappeared the instant the file was opened — before `write()` was even called.

Now `"a"`, which keeps what is there.

In [ ]:
with open(path, "a") as f:
    f.write("added later\n")
    f.write("and one more\n")

show(path)

## 2.2 Why `with` matters

This is the demo worth stopping for.

Below we open a file **without** `with`, write to it, and then try to read it
*before* closing it.

In [ ]:
leaky = DEMO / "leaky.txt"

f = open(leaky, "w")
f.write("some important data")

# read it from a second, separate handle -- before we close the first one
print("while still open :", repr(open(leaky).read()))

f.close()
print("after close      :", repr(open(leaky).read()))

**The file was empty until `close()` ran.**

What you write goes into a buffer in memory first. It only reaches the disk when the file
is closed. Forget `close()` — or crash before reaching it — and the data is simply lost.

`with` closes the file for you, **even if the code inside crashes**. Proof:

In [ ]:
safe = DEMO / "safe.txt"

try:
    with open(safe, "w") as f:
        f.write("written before the crash")
        raise ValueError("something went wrong!")
except ValueError as e:
    print("caught:", e)

# the crash happened INSIDE the with-block, yet the data is on disk
show(safe)

That is the whole argument. **Always use `with`.**

## 2.3 Reading, three ways

In [ ]:
products = DEMO / "products.txt"

with open(products, "w") as f:
    for name in ["Laptop", "Mouse", "Cable", "Hub", "Screen"]:
        f.write(name + "\n")

show(products)

In [ ]:
# (a) the whole file as one string
with open(products) as f:
    text = f.read()

print(repr(text))
print("type:", type(text), "| length:", len(text))

In [ ]:
# (b) a list of lines -- note the \n still attached to each one
with open(products) as f:
    lines = f.readlines()

print(lines)

In [ ]:
# (c) one line at a time -- works on a file of any size
with open(products) as f:
    for n, line in enumerate(f, start=1):
        print(n, repr(line), "->", repr(line.strip()))

`.strip()` removes the newline at the end. Forget it and your values quietly carry a
`\n` around with them — a bug that is very hard to see.

In [ ]:
with open(products) as f:
    raw     = [line for line in f]

with open(products) as f:
    cleaned = [line.strip() for line in f]

print("raw    :", raw)
print("cleaned:", cleaned)
print()
print('Is "Mouse" in raw?    ', "Mouse" in raw)        # False! -- it is "Mouse\n"
print('Is "Mouse" in cleaned?', "Mouse" in cleaned)

---
# 3 · CSV — the language of business data

A CSV is **plain text**. One line per row, commas between the fields, and usually a
header line naming the columns.

Excel opens it, but a CSV holds no colours, no formulas and no types.

In [ ]:
import csv

catalogue = [
    {"name": "Laptop", "price": 899.0, "stock": 4},
    {"name": "Mouse",  "price": 24.5,  "stock": 31},
    {"name": "Cable",  "price": 9.99,  "stock": 0},
    {"name": "Hub",    "price": 149.0, "stock": 7},
]

csv_path = DEMO / "catalogue.csv"

with open(csv_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "price", "stock"])
    writer.writeheader()
    writer.writerows(catalogue)

show(csv_path)

That is the entire file — text you could type by hand.

Open `demo/catalogue.csv` in the VS Code explorer. It is just text.

## 3.1 Reading it back

In [ ]:
with open(csv_path, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    rows = list(reader)

for row in rows:
    print(row)

Each row came back as a **dictionary** — your list of dictionaries from Session 3,
loaded from disk.

## 3.2 The catch: everything is text

In [ ]:
first = rows[0]

print("price value :", repr(first["price"]))
print("price type  :", type(first["price"]))
print()
print("Is it more than 500?", first["price"] > "500")   # comparing TEXT, not numbers

In [ ]:
# THIS CELL IS MEANT TO FAIL
# EXPECT: TypeError
total = 0
for row in rows:
    total = total + row["price"] * row["stock"]
print(total)

`TypeError` — you cannot multiply two strings together.

The fix is to convert as you read. A CSV has no idea what a number is, so you must say so.

In [ ]:
total = 0
for row in rows:
    total = total + float(row["price"]) * int(row["stock"])

print("stock value:", round(total, 2))

## 3.3 The comma problem

What happens when a value *contains* a comma?

In [ ]:
tricky = DEMO / "tricky.csv"

# written properly, with the csv module
with open(tricky, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "note"])
    w.writerow(["Mouse, wireless", "two buttons"])

show(tricky)

The `csv` module wrapped the value in quotes automatically, so it survives.

Here is the same data written **by hand** with string joining — the mistake everyone makes
at least once.

In [ ]:
broken = DEMO / "broken.csv"

rows_out = [["name", "note"], ["Mouse, wireless", "two buttons"]]
with open(broken, "w", encoding="utf-8") as f:
    for r in rows_out:
        f.write(",".join(r) + "\n")

show(broken)

with open(broken, newline="", encoding="utf-8") as f:
    for r in csv.reader(f):
        print(len(r), "fields ->", r)

Two columns became three. The data is silently corrupted.

**Never build a CSV by joining strings. Use the `csv` module.**

---
# 4 · JSON — the language of the web

JSON looks almost exactly like a Python dictionary written out as text.

Unlike CSV it keeps **types**, and it can hold **nested** data.

In [ ]:
import json

json_path = DEMO / "catalogue.json"

with open(json_path, "w", encoding="utf-8") as f:
    json.dump(catalogue, f, indent=2)

show(json_path)

`indent=2` is what makes it readable. Without it the whole file is one long line.

In [ ]:
with open(json_path, encoding="utf-8") as f:
    loaded = json.load(f)

print(loaded[0])
print()
print("price value:", repr(loaded[0]["price"]))
print("price type :", type(loaded[0]["price"]))     # float, not str!
print("stock type :", type(loaded[0]["stock"]))     # int

**The types survived.** The CSV gave back `'899.0'`; JSON gives back `899.0`.

So the arithmetic that failed earlier just works:

In [ ]:
total = sum(row["price"] * row["stock"] for row in loaded)
print("stock value:", round(total, 2))

## 4.1 Four names, easy to mix up

| Function | Direction | Works with |
|---|---|---|
| `json.dump(obj, f)` | out | a **file** |
| `json.load(f)` | in | a **file** |
| `json.dumps(obj)` | out | a **string** |
| `json.loads(s)` | in | a **string** |

The `s` stands for *string*. That one letter is the whole difference.

In [ ]:
text = json.dumps({"name": "Hub", "price": 149.0})
print("dumps ->", repr(text), type(text))

back = json.loads(text)
print("loads ->", back, type(back))

## 4.2 Shapes a CSV cannot hold

In [ ]:
order = {
    "id": 1042,
    "customer": {"name": "Anna", "city": "Vaduz"},
    "items": [
        {"product": "Laptop", "qty": 1},
        {"product": "Mouse",  "qty": 2},
    ],
    "paid": True,
}

order_path = DEMO / "order.json"
with open(order_path, "w", encoding="utf-8") as f:
    json.dump(order, f, indent=2)

show(order_path)

In [ ]:
with open(order_path, encoding="utf-8") as f:
    o = json.load(f)

print("city        :", o["customer"]["city"])
print("second item :", o["items"][1]["product"])
print("paid        :", o["paid"], type(o["paid"]))    # a real boolean

A dictionary inside a dictionary, and a list of dictionaries inside that.
Each bracket takes you one level deeper.

Try writing that as a flat CSV — you cannot, and that is exactly when you reach for JSON.

---
# 5 · Error handling

This is the part that turns a script into a program.

Everything below is a **normal situation**, not a bug in your code: the file is missing,
the file is damaged, the user typed a word where a number belongs. Your program has to
survive all of it.

## 5.1 The file that is not there

In [ ]:
# THIS CELL IS MEANT TO FAIL
# EXPECT: FileNotFoundError
with open(DEMO / "does_not_exist.json") as f:
    data = json.load(f)
print(data)

Read the traceback **from the bottom up**. The last line names the error and the file.

On the very first run of any program, the data file never exists. So this is not an edge
case — it is the normal first experience of every user.

In [ ]:
try:
    with open(DEMO / "does_not_exist.json") as f:
        data = json.load(f)
except FileNotFoundError:
    print("No file yet -- starting with an empty list.")
    data = []

print("data =", data)
print("The program is still running.")

## 5.2 All four clauses

`try` / `except` / `else` / `finally`. Watch the order in which they run.

In [ ]:
def read_count(path):
    print(f"\n>>> read_count({path.name})")
    try:
        print("  try      : opening")
        with open(path, encoding="utf-8") as f:
            rows = json.load(f)
    except FileNotFoundError:
        print("  except   : no such file")
        return 0
    else:
        print("  else     : nothing went wrong")
        return len(rows)
    finally:
        print("  finally  : runs either way")


print("result:", read_count(json_path))
print("result:", read_count(DEMO / "nope.json"))

Look closely at the second call: `finally` printed **after** `except`, and still ran even
though the function was returning.

- `try` — the risky part, kept as short as you can
- `except` — what to do about one *named* problem
- `else` — only when nothing went wrong
- `finally` — always, no matter what. Cleanup belongs here.

## 5.3 The most expensive mistake: a bare `except`

Here is a function with a typo in it. Watch what the bare `except` does.

In [ ]:
def count_products_BAD(path):
    try:
        with open(path, encoding="utf-8") as f:
            rows = json.load(f)
        return len(rowss)          # <-- typo: rowss
    except:                        # <-- catches absolutely everything
        return 0


print("count:", count_products_BAD(json_path))
print()
print("The file has 4 products. The function says 0, and reports no problem at all.")

There is a `NameError` in that function and **you would never find out**.
The bare `except` swallowed it and returned a plausible-looking wrong answer.

Now the same function, catching only what it can actually handle:

In [ ]:
# THIS CELL IS MEANT TO FAIL
# EXPECT: NameError
def count_products_GOOD(path):
    try:
        with open(path, encoding="utf-8") as f:
            rows = json.load(f)
        return len(rowss)                  # same typo
    except FileNotFoundError:              # only the problem we expect
        return 0


print("count:", count_products_GOOD(json_path))

`NameError: name 'rowss' is not defined` — exactly what we needed to see.

**Never write a bare `except:`.** Name the error you expect. Anything you did not expect
is a bug, and a bug should be loud.

The same goes for `except Exception:` — it is only slightly narrower.

## 5.4 A damaged file

The file exists, but something is wrong inside it. This happens when a program was
interrupted halfway through writing.

In [ ]:
corrupt = DEMO / "corrupt.json"
corrupt.write_text('[{"name": "Laptop", "price": 899.0', encoding="utf-8")   # cut off

show(corrupt)

In [ ]:
# THIS CELL IS MEANT TO FAIL
# EXPECT: JSONDecodeError
with open(corrupt, encoding="utf-8") as f:
    json.load(f)

In [ ]:
# an empty file is damaged too, in its own way
empty = DEMO / "empty.json"
empty.write_text("", encoding="utf-8")

try:
    with open(empty, encoding="utf-8") as f:
        json.load(f)
except json.JSONDecodeError as e:
    print("JSONDecodeError:", e)

Note `as e` — it gives you the error object, so you can print the detail or log it.

## 5.5 Catching more than one thing

A real loader has to survive **both** problems: missing *and* damaged.

In [ ]:
def load_catalogue(path):
    try:
        with open(path, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        print(f"  [{path.name}] no file -- starting empty")
        return []
    except json.JSONDecodeError as e:
        print(f"  [{path.name}] damaged ({e.msg}) -- starting empty")
        return []


print("good file :", load_catalogue(json_path))
print()
print("missing   :", load_catalogue(DEMO / "nope.json"))
print()
print("corrupt   :", load_catalogue(corrupt))

Three very different situations, one function, and the program never stops.

You can also group errors that deserve the same treatment:

```python
except (FileNotFoundError, json.JSONDecodeError):
    return []
```

## 5.6 Bad data inside a good file

The file loads fine, but a row is missing a field.

In [ ]:
messy = DEMO / "messy.json"

with open(messy, "w", encoding="utf-8") as f:
    json.dump([
        {"name": "Laptop", "price": 899.0, "stock": 4},
        {"name": "Mouse",  "price": 24.5},                              # no stock
        {"name": "Cable",  "price": "nine ninety nine", "stock": 0},    # price is text
    ], f, indent=2)

show(messy)

In [ ]:
# THIS CELL IS MEANT TO FAIL
# EXPECT: KeyError
rows = json.loads(messy.read_text(encoding="utf-8"))
for row in rows:
    print(row["name"], row["stock"])

`KeyError: 'stock'` — the row simply has no such field.

Two ways to deal with it. Use `.get()` when a missing field is **acceptable**:

In [ ]:
for row in rows:
    print(f"{row['name']:8} stock = {row.get('stock', 'unknown')}")

Use `try` / `except` when a missing or unusable field means **skip this row**:

In [ ]:
total = 0
skipped = []

for row in rows:
    try:
        total += float(row["price"]) * int(row["stock"])
    except KeyError as e:
        skipped.append(f"{row.get('name', '?')}: missing field {e}")
    except ValueError as e:
        skipped.append(f"{row.get('name', '?')}: bad number ({e})")

print("stock value:", round(total, 2))
print("skipped rows:")
for s in skipped:
    print("  -", s)

The program produced an answer **and** told you exactly what it had to ignore.
That is what good error handling looks like: never silent, never fatal.

## 5.7 Bad input from a person

In [ ]:
def to_int(text, fallback=0):
    try:
        return int(text)
    except ValueError:
        print(f"  '{text}' is not a whole number -- using {fallback}")
        return fallback


for attempt in ["42", "  7 ", "abc", "3.5", ""]:
    print(repr(attempt), "->", to_int(attempt))

Note that `"3.5"` fails too: `int()` will not parse a decimal point from a string.
`int(float("3.5"))` is what you want there.

## 5.8 Raising your own

Sometimes the right answer is to refuse and say why.

In [ ]:
def add_product(rows, name, price):
    if not name:
        raise ValueError("a product needs a name")
    if price < 0:
        raise ValueError(f"price cannot be negative (got {price})")
    rows.append({"name": name, "price": price, "stock": 0})
    return rows


shop = []
add_product(shop, "Hub", 149.0)
print("ok  :", shop)

for bad_name, bad_price in [("", 10.0), ("Ghost", -5.0)]:
    try:
        add_product(shop, bad_name, bad_price)
    except ValueError as e:
        print("refused:", e)

print("shop is still clean:", shop)

A clear error at the moment the bad data arrives beats a confusing one three functions later.

---
# 6 · The build: a catalogue that survives

Everything from today, in one small program.

In [ ]:
SHOP_FILE = DEMO / "shop.json"


def load():
    try:
        with open(SHOP_FILE, encoding="utf-8") as f:
            return json.load(f)
    except FileNotFoundError:
        return []
    except json.JSONDecodeError:
        print("warning: shop.json was damaged, starting empty")
        return []


def save(rows):
    with open(SHOP_FILE, "w", encoding="utf-8") as f:
        json.dump(rows, f, indent=2)


def report(rows):
    if not rows:
        print("(the shop is empty)")
        return
    for row in rows:
        mark = "" if row["stock"] > 0 else "   SOLD OUT"
        print(f"  {row['name']:8} {row['price']:>8.2f} EUR   x{row['stock']}{mark}")
    value = sum(r["price"] * r["stock"] for r in rows)
    print(f"  {'':8} {'':>8}       total {value:.2f} EUR")


print("first run -- no file exists yet")
shop = load()
report(shop)

In [ ]:
shop.append({"name": "Laptop", "price": 899.0, "stock": 4})
shop.append({"name": "Mouse",  "price": 24.5,  "stock": 31})
shop.append({"name": "Cable",  "price": 9.99,  "stock": 0})
save(shop)

report(shop)
ls()

Now the moment that matters. **Pretend the program ended** — throw the variable away
completely — then load it back from disk.

In [ ]:
del shop                      # the list no longer exists in memory

try:
    report(shop)
except NameError as e:
    print("in memory:", e)

print()
print("from disk:")
shop = load()                 # read it back off the disk
report(shop)

**That is data that survives.**

Restart the kernel and run only the setup cell and this one — the shop is still there,
because it lives in `demo/shop.json`, not in memory.

In [ ]:
show(SHOP_FILE)

And it survives a damaged file too:

In [ ]:
SHOP_FILE.write_text('[{"name": "Laptop", ', encoding="utf-8")   # break it on purpose

shop = load()        # warns, does not crash
report(shop)

print()
print("recovering...")
shop = [{"name": "Laptop", "price": 899.0, "stock": 4}]
save(shop)
report(shop)

---
# 7 · Paths that work on every machine

Windows separates folders with `\`, macOS and Linux with `/`. Writing either one by hand
means your code breaks on somebody else's laptop.

`pathlib` handles it for you.

In [ ]:
from pathlib import Path

folder = DEMO / "exports"          # the / operator joins paths
folder.mkdir(exist_ok=True)        # exist_ok: do not complain if it is already there

target = folder / "report.txt"
target.write_text("Stock report\n============\n", encoding="utf-8")

print("path       :", target)
print("exists     :", target.exists())
print("name       :", target.name)
print("suffix     :", target.suffix)
print("parent     :", target.parent)
print("absolute   :", target.resolve())
print()
print(target.read_text(encoding="utf-8"))

In [ ]:
# .exists() asks politely; try/except is still the safer habit,
# because a file can disappear between the question and the answer
maybe = DEMO / "maybe.txt"

if maybe.exists():
    print(maybe.read_text(encoding="utf-8"))
else:
    print(f"{maybe.name} is not there -- and we did not crash")

In [ ]:
# everything we created today
print("Files in demo/:")
for p in sorted(DEMO.rglob("*")):
    kind = "dir " if p.is_dir() else "file"
    size = "" if p.is_dir() else f"{p.stat().st_size:>6} bytes"
    print(f"  {kind} {str(p.relative_to(DEMO)):24} {size}")

---
# What you just saw

| | |
|---|---|
| `with open(path, mode)` | `"r"` read, `"w"` write **and erase**, `"a"` append |
| `with` | closes the file even when the code inside crashes |
| `.strip()` | removes the `\n` that hides inside every line you read |
| `csv.DictReader` / `DictWriter` | rows as dictionaries — but **every value is text** |
| `json.dump` / `load` | keeps types and nesting; `dumps` / `loads` for strings |
| `try` / `except` | name the error you expect, never a bare `except:` |
| `else` / `finally` | only-if-fine, and always |
| `.get(key, default)` | when a missing field is acceptable |
| `raise ValueError(...)` | refuse bad data at the moment it arrives |
| `pathlib.Path` | paths that work on every operating system |

### The one rule to remember

> Catch the errors you can do something about. Let the rest be loud.

A bare `except:` turns a bug into a wrong answer that nobody notices.

---

**Next, in part two:** that `catalogue.json` stops being a private file.
We put it on the web with FastAPI, so any program on the internet can ask for it.